# 02 · Features, Scaling, and the Training Kernel

This notebook follows the 24 training examples from host features to the quantum kernel the quantum-kernel SVM is trained on. It shows how the four host features are computed, re-runs the documented scaling step and checks that it reproduces the versioned scaler, draws the four-qubit feature map and one compute-uncompute pair circuit, and computes the 24 x 24 training kernel by exact statevector simulation with the diagnostics the protocol requires. Every number here is either classical preprocessing or **exact simulation**; no IBM Quantum service is contacted, and nothing is tuned.

**Inputs**
- `configs/benchmark-split.json` (frozen split manifest)
- `configs/feature-scaler.json` (versioned scaler, used as the reference)
- `data/processed/benchmark-sample.csv` (local, produced by `scripts/prepare_benchmark.py` in notebook 01)
- `docs/benchmark-protocol.md` (recorded values the checks compare against)
- `artifacts/benchmark/training-angles.npy`, `artifacts/benchmark/training-kernel.npy` (optional; compared only if a training run already left them)

**Outputs**
- `configs/feature-scaler.json` (regenerated by `scripts/scale_benchmark.py`; must be identical to the versioned file)
- `data/processed/benchmark-scaled.csv` (local, Git-ignored angles for the 36 sample rows)

Run after: `01-dataset-and-split.ipynb`

In [1]:
import os
import sys
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src").is_dir() and (p / "configs").is_dir()
)
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import hashlib
import importlib.util
import json
import math
import re
from collections import Counter

import numpy as np
import pandas as pd
import qiskit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector

from src.features.host_features import host_features, normalize_host
from src.features.scaling import ANGLE_MAX, AngleScaler
from src.quantum import kernel as qkernel

MANIFEST = json.loads(Path("configs/benchmark-split.json").read_text(encoding="utf-8"))
FEATURES = MANIFEST["features"]
PROTOCOL = Path("docs/benchmark-protocol.md").read_text(encoding="utf-8")
print("Repository root located; working directory set to it.")
print(
    f"Python {sys.version.split()[0]}, qiskit {qiskit.__version__}, numpy {np.__version__}, pandas {pd.__version__}"
)
print("Frozen features (one per qubit, in order):", FEATURES)

Repository root located; working directory set to it.
Python 3.12.10, qiskit 2.6.1, numpy 2.5.4, pandas 3.1.0
Frozen features (one per qubit, in order): ['host_entropy', 'host_label_count', 'host_digit_ratio', 'host_hyphen_count']


## 1. Host features

The four frozen features use only the host: lowercased, with a trailing dot, a port, and one leading `www.` removed (`normalize_host`). Paths, queries, and the scheme are ignored on purpose. In PhiUSIIL every legitimate URL is a bare `https://www.<host>` homepage, so those parts would let a model learn how the dataset was collected rather than anything about the host.

`host_features` returns eight candidate features; the split manifest froze four of them. Below it runs on a made-up host that is not from the dataset.

In [2]:
example = "WWW.Secure-Login2.Example-Shop.co.uk."
host = normalize_host(example)
values = host_features(host)
print(f"raw input       : {example}")
print(f"normalized host : {host}")
for name in FEATURES:
    print(f"  {name:18s} = {values[name]:.6f}")

# The same numbers by hand, to make each definition explicit.
counts = Counter(host)
by_hand = {
    "host_entropy": -sum(c / len(host) * math.log2(c / len(host)) for c in counts.values()),
    "host_label_count": float(len(host.split("."))),
    "host_digit_ratio": sum(ch.isdigit() for ch in host) / len(host),
    "host_hyphen_count": float(host.count("-")),
}
assert all(math.isclose(values[k], by_hand[k], rel_tol=0, abs_tol=1e-12) for k in FEATURES)
print(
    "OK: host_features matches the hand-computed definitions (Shannon entropy in bits, label count, digit share, hyphens)."
)

raw input       : WWW.Secure-Login2.Example-Shop.co.uk.
normalized host : secure-login2.example-shop.co.uk
  host_entropy       = 4.077820
  host_label_count   = 4.000000
  host_digit_ratio   = 0.031250
  host_hyphen_count  = 2.000000
OK: host_features matches the hand-computed definitions (Shannon entropy in bits, label count, digit share, hyphens).


### Training rows, aggregated per class

The local sample holds the 24 training and 12 test rows chosen in notebook 01. Hosts are never printed; rows are identified only by `row_index` and the manifest's `group_hash`. First we confirm that the sample matches the manifest and that `host_features` reproduces every stored feature value.

In [3]:
sample = pd.read_csv("data/processed/benchmark-sample.csv")
train = sample[sample["partition"] == "train"].reset_index(drop=True)
test = sample[sample["partition"] == "test"].reset_index(drop=True)

for name, rows in (("train", train), ("test", test)):
    expected = [(r["row_index"], r["is_phishing"]) for r in MANIFEST[name]]
    assert list(zip(rows["row_index"], rows["is_phishing"], strict=True)) == expected, (
        f"{name} rows differ from the manifest"
    )
print(
    f"OK: sample rows, order, and labels match the manifest (train {len(train)}, test {len(test)})."
)

recomputed = pd.DataFrame([host_features(normalize_host(h)) for h in sample["host"]])[FEATURES]
assert np.array_equal(recomputed.to_numpy(), sample[FEATURES].to_numpy()), (
    "stored features differ from host_features"
)
print(
    f"OK: all {len(sample)} stored feature rows are reproduced exactly by host_features(normalize_host(host))."
)

labels = train["is_phishing"].map({0: "legitimate", 1: "phishing"}).rename("class")
summary = train[FEATURES].groupby(labels).agg(["mean", "std", "min", "max"]).T
summary.index.names = ["feature", "statistic"]
print(f"Training rows per class: {labels.value_counts().sort_index().to_dict()}")
summary.round(4)

OK: sample rows, order, and labels match the manifest (train 24, test 12).
OK: all 36 stored feature rows are reproduced exactly by host_features(normalize_host(host)).
Training rows per class: {'legitimate': 12, 'phishing': 12}


class                        legitimate  phishing
feature           statistic                      
host_entropy      mean           3.4828    3.5220
                  std            0.3652    0.6054
                  min            2.6258    2.3219
                  max            3.8454    4.5739
host_label_count  mean           2.1667    2.4167
                  std            0.3892    0.5149
                  min            2.0000    2.0000
                  max            3.0000    3.0000
host_digit_ratio  mean           0.0000    0.0359
                  std            0.0000    0.0655
                  min            0.0000    0.0000
                  max            0.0000    0.1558
host_hyphen_count mean           0.0833    1.1667
                  std            0.2887    2.1249
                  min            0.0000    0.0000
                  max            1.0000    6.0000

In [4]:
share_zero = train[FEATURES].eq(0).groupby(labels).mean().T
print("Share of training rows whose feature value is exactly 0, per class:")
share_zero.round(3)

Share of training rows whose feature value is exactly 0, per class:


class,legitimate,phishing
host_entropy,0.000,0.000
host_label_count,0.000,0.000
host_digit_ratio,1.000,0.750
host_hyphen_count,0.917,0.667


## 2. Training-only angle scaling

`scripts/scale_benchmark.py` is the documented step. It fits a min-max scaler on the 24 training rows only and maps each feature to an angle in `[0, pi/2]`. Test and inference values outside the training range are clipped and counted. A feature that is constant in training would map to 0; none is.

Why `pi/2` and not `pi`: the feature map applies the single-qubit phase `P(2 * x)`. With `x` up to `pi`, the smallest and largest training values would receive phases `0` and `2*pi`, which are the same gate. Stopping at `pi/2` keeps the phase in `[0, pi]`, so the extremes stay distinguishable.

The cell keeps the versioned scaler in memory, runs the script, and the check below compares the two.

In [5]:
scaler_path = Path("configs/feature-scaler.json")
scaled_path = Path("data/processed/benchmark-scaled.csv")


def normalized_text(path):
    return path.read_text(encoding="utf-8").replace("\r\n", "\n")


# sha256 of the committed configs/feature-scaler.json (LF line endings, UTF-8). Hard-coded so the check
# is anchored to the published file even without git (for example, from a zip download).
COMMITTED_SCALER_SHA256_LF = "11cacc3917a1cf2a865d0b34e79a5c8c6e48e5adc42c7ad5886a92fe31b8f29b"


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


versioned_text = normalized_text(scaler_path)
versioned = json.loads(versioned_text)
if sha256_text(versioned_text) != COMMITTED_SCALER_SHA256_LF:
    print(
        "WARNING: the scaler on disk before this run differs from the committed one; the check below decides."
    )

spec = importlib.util.spec_from_file_location(
    "scale_benchmark", ROOT / "scripts" / "scale_benchmark.py"
)
scale_benchmark = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scale_benchmark)
return_code = scale_benchmark.main()
assert return_code == 0, f"scale_benchmark.py returned {return_code}"

Training range per feature:
  host_entropy: 2.3219 to 4.5739
  host_label_count: 2.0000 to 3.0000
  host_digit_ratio: 0.0000 to 0.1558
  host_hyphen_count: 0.0000 to 6.0000
Test values clipped: {'host_entropy': 1, 'host_label_count': 2, 'host_digit_ratio': 1, 'host_hyphen_count': 0}
Scaler: configs\feature-scaler.json


### Replication check: scaler

In [6]:
regenerated_text = normalized_text(scaler_path)
regenerated = json.loads(regenerated_text)
assert regenerated == versioned, (
    "regenerated scaler differs from the versioned configs/feature-scaler.json"
)
assert regenerated_text == versioned_text, "scaler content matches but its formatting differs"
print(f"OK: {scaler_path.as_posix()} regenerated identically (same text, line endings aside).")
assert sha256_text(regenerated_text) == COMMITTED_SCALER_SHA256_LF, (
    "regenerated scaler differs from the committed file"
)
print(
    f"OK: matches the committed scaler (sha256 {COMMITTED_SCALER_SHA256_LF[:8]}..., LF line endings)."
)

assert versioned["angle_max"] == math.pi / 2 == ANGLE_MAX
assert versioned["features"] == FEATURES and not any(versioned["constant"])
assert "train partition" in versioned["fitted_on"]
train_matrix = train[FEATURES].to_numpy()
assert versioned["minima"] == train_matrix.min(axis=0).tolist()
assert versioned["maxima"] == train_matrix.max(axis=0).tolist()
print(
    "OK: angle_max = pi/2; minima and maxima equal the 24 training rows' extremes; no constant feature."
)

# Compare with the preprocessing table recorded in docs/benchmark-protocol.md.
table = {}
for name in FEATURES:
    m = re.search(rf"\| `{name}` \| ([0-9.]+) \| ([0-9.]+) \| (\d+) \|", PROTOCOL)
    assert m, f"no protocol row for {name}"
    table[name] = (float(m[1]), float(m[2]), int(m[3]))
for i, name in enumerate(FEATURES):
    low, high, clipped = table[name]
    assert (
        abs(versioned["minima"][i] - low) <= 5e-5 and abs(versioned["maxima"][i] - high) <= 5e-5
    ), name
    assert versioned["clipped_values"]["test"][name] == clipped, name
print(
    "OK: training ranges (4 decimals) and test clipping counts match the protocol's preprocessing table."
)
pd.DataFrame(table, index=["documented min", "documented max", "documented test clipped"]).T

OK: configs/feature-scaler.json regenerated identically (same text, line endings aside).
OK: matches the committed scaler (sha256 11cacc39..., LF line endings).
OK: angle_max = pi/2; minima and maxima equal the 24 training rows' extremes; no constant feature.
OK: training ranges (4 decimals) and test clipping counts match the protocol's preprocessing table.


,documented min,documented max,documented test clipped
host_entropy,2.3219,4.5739,1.0
host_label_count,2.0000,3.0000,2.0
host_digit_ratio,0.0000,0.1558,1.0
host_hyphen_count,0.0000,6.0000,0.0


In [7]:
scaler = AngleScaler.from_dict(versioned)
train_angles, train_clipped = scaler.transform(train[FEATURES])
test_angles, test_clipped = scaler.transform(test[FEATURES])
assert train_clipped == [0, 0, 0, 0], "training values must never be clipped"
assert (train_angles.min(axis=0) == 0).all()
assert np.allclose(train_angles.max(axis=0), ANGLE_MAX, rtol=0, atol=1e-15)
print("OK: training angles span exactly [0, pi/2] per feature, with no training value clipped.")

scaled = pd.read_csv(scaled_path)
assert np.array_equal(scaled[scaled["partition"] == "train"][FEATURES].to_numpy(), train_angles)
assert np.array_equal(scaled[scaled["partition"] == "test"][FEATURES].to_numpy(), test_angles)
print(f"OK: {scaled_path.as_posix()} holds the same angles as AngleScaler.transform.")

saved_angles = Path("artifacts/benchmark/training-angles.npy")
if saved_angles.exists():
    assert np.array_equal(np.load(saved_angles), train_angles), (
        "training angles differ from the trained artifacts"
    )
    print(f"OK: angles equal {saved_angles.as_posix()} from the training run.")
else:
    print(
        f"SKIPPED: {saved_angles.as_posix()} not present yet (created by scripts/train_benchmark.py)."
    )

print(
    "host_label_count angles in training:",
    sorted(set(train_angles[:, 1].round(6).tolist())),
    "-> a binary 0 / pi/2 feature",
)

OK: training angles span exactly [0, pi/2] per feature, with no training value clipped.
OK: data/processed/benchmark-scaled.csv holds the same angles as AngleScaler.transform.
SKIPPED: artifacts/benchmark/training-angles.npy not present yet (created by scripts/train_benchmark.py).
host_label_count angles in training: [0.0, 1.570796] -> a binary 0 / pi/2 feature


### Test values outside the training range

Clipping is part of the frozen preprocessing. It is shown here for transparency and changes nothing. Test rows are identified by `row_index` and `group_hash` only.

In [8]:
low, high = np.array(scaler.minima), np.array(scaler.maxima)
raw = test[FEATURES].to_numpy()
outside = (raw < low) | (raw > high)
group_hash = {r["row_index"]: r["group_hash"] for r in MANIFEST["test"]}
rows = []
for i, j in zip(*np.nonzero(outside), strict=True):
    rows.append(
        {
            "row_index": int(test.loc[i, "row_index"]),
            "group_hash": group_hash[test.loc[i, "row_index"]],
            "is_phishing": int(test.loc[i, "is_phishing"]),
            "feature": FEATURES[j],
            "raw value": raw[i, j],
            "training range": f"[{low[j]:.4f}, {high[j]:.4f}]",
            "angle after clip": test_angles[i, j],
        }
    )
clipped_table = pd.DataFrame(rows)
assert dict(zip(FEATURES, test_clipped, strict=True)) == versioned["clipped_values"]["test"]
assert len(clipped_table) == sum(test_clipped)
print(
    f"OK: {len(clipped_table)} clipped test values, matching the counts saved in the scaler: {dict(zip(FEATURES, test_clipped, strict=True))}."
)
clipped_table.round(4)

OK: 4 clipped test values, matching the counts saved in the scaler: {'host_entropy': 1, 'host_label_count': 2, 'host_digit_ratio': 1, 'host_hyphen_count': 0}.


,row_index,group_hash,is_phishing,feature,raw value,training range,angle after clip
0,41538,e51ce3301401094b,1,host_label_count,4.0000,"[2.0000, 3.0000]",1.5708
1,121258,62c0d81c126f3930,1,host_entropy,1.9183,"[2.3219, 4.5739]",0.0000
2,235425,0efd30f928e7c167,1,host_label_count,5.0000,"[2.0000, 3.0000]",1.5708
3,88834,79a39c435af85683,1,host_digit_ratio,0.2381,"[0.0000, 0.1558]",1.5708


## 3. The circuit

The encoding `U(x)` is Qiskit's `zz_feature_map(feature_dimension=4, reps=1, entanglement="linear", alpha=2.0)`. It applies a Hadamard and `P(2 x_i)` on each qubit, then, for each neighbouring pair `(0,1)`, `(1,2)`, `(2,3)`, a `CX`, `P(2 (pi - x_i)(pi - x_j))`, and another `CX`. The kernel value `K(x, y) = |<0000| U(y)^dagger U(x) |0000>|^2` is the probability of measuring `0000` after the compute-uncompute pair circuit. Full description: [docs/quantum-circuit.md](../docs/quantum-circuit.md).

The circuit has no trainable gate angles; it is a fixed data encoding. The counts below are logical, before transpilation for any backend.

In [9]:
fmap = qkernel.feature_map()
print("Feature map settings:", qkernel.FEATURE_MAP_SETTINGS)
print(fmap.draw(output="text", fold=-1))
fmap_ops = dict(fmap.count_ops())
print("Logical gate counts:", fmap_ops, "| depth:", fmap.depth())
assert fmap_ops == {"h": 4, "p": 7, "cx": 6}, fmap_ops
print("OK: one feature map = 4 H, 7 phase, 6 CX, as stated in docs/quantum-circuit.md.")

Feature map settings: {'feature_dimension': 4, 'reps': 1, 'entanglement': 'linear', 'alpha': 2.0}
   ┌───┐┌───────────┐                                                                                                                              
0: ┤ H ├┤ P(2*x[0]) ├──■────────────────────────────────────■──────────────────────────────────────────────────────────────────────────────────────
   ├───┤├───────────┤┌─┴─┐┌──────────────────────────────┐┌─┴─┐                                                                                    
1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P((-π + x[0])*(-π + x[1])*2) ├┤ X ├──■────────────────────────────────────■────────────────────────────────────────────
   ├───┤├───────────┤└───┘└──────────────────────────────┘└───┘┌─┴─┐┌──────────────────────────────┐┌─┴─┐                                          
2: ┤ H ├┤ P(2*x[2]) ├──────────────────────────────────────────┤ X ├┤ P((-π + x[1])*(-π + x[2])*2) ├┤ X ├──■────────────────────────────────────■──
   ├───┤├─────

In [10]:
# One pair: training positions 0 and 1 (both legitimate). The angles are bound, so the gates show numbers.
pair = qkernel.pair_circuit(train_angles[0], train_angles[1])
print(pair.draw(output="text", fold=160))
pair_ops = dict(pair.count_ops())
print("Logical gate counts:", pair_ops, "| depth:", pair.depth())
assert {k: pair_ops[k] for k in ("h", "p", "cx", "measure")} == {
    "h": 8,
    "p": 14,
    "cx": 12,
    "measure": 4,
}, pair_ops
print(
    "OK: one pair circuit = 8 H, 14 phase, 12 CX, 4 measurements, as stated in docs/quantum-circuit.md."
)

        ┌───┐┌──────────┐                                                                                                                                »
     0: ┤ H ├┤ P(1.873) ├──■─────────────────■──────────────────────────────────────────────────────────────────────────────────────────────────────■────»
        ├───┤└─┬──────┬─┘┌─┴─┐┌───────────┐┌─┴─┐                                                                                                  ┌─┴─┐  »
     1: ┤ H ├──┤ P(0) ├──┤ X ├┤ P(13.855) ├┤ X ├──■─────────────────■────────────────────────────────────────────────────■────────────────────■───┤ X ├──»
        ├───┤  ├──────┤  └───┘└───────────┘└───┘┌─┴─┐┌───────────┐┌─┴─┐                                                ┌─┴─┐  ┌────────────┐┌─┴─┐┌┴───┴─┐»
     2: ┤ H ├──┤ P(0) ├─────────────────────────┤ X ├┤ P(19.739) ├┤ X ├──■─────────────────■────■──────────────────■───┤ X ├──┤ P(-19.739) ├┤ X ├┤ P(0) ├»
        ├───┤  ├──────┤                         └───┘└───────────┘└───

## 4. The exact training kernel

`statevector_kernel` simulates `U(x)|0000>` for each of the 24 training examples and takes the squared overlaps. The quantum-kernel SVM is fitted to this kernel; `scripts/train_benchmark.py` makes exactly this call. Hardware is used later, only for the test kernel. `training_kernel_diagnostics` is the protocol's validity and concentration check.

In [11]:
K = qkernel.statevector_kernel(train_angles)
diag = qkernel.training_kernel_diagnostics(K)
y = train["is_phishing"].to_numpy()
off = ~np.eye(len(K), dtype=bool)
same = y[:, None] == y[None, :]
within = float(K[same & off].mean())
cross = float(K[~same].mean())

print("Training kernel, exact statevector simulation, shape", diag["shape"])
for key in ("min_eigenvalue", "max_asymmetry", "max_diagonal_error"):
    print(f"  {key:24s} {diag[key]: .2e}")
print(
    f"  {'eigenvalue_check_passed':24s} {diag['eigenvalue_check_passed']} (tolerance -{qkernel.EIGENVALUE_TOLERANCE:g})"
)
for key in ("off_diagonal_mean", "off_diagonal_std", "off_diagonal_min", "off_diagonal_max"):
    print(f"  {key:24s} {diag[key]:.4f}")
print(f"  {'identical_example_pairs':24s} {diag['identical_example_pairs']}")
print(f"Mean similarity within a class (off-diagonal): {within:.4f}")
print(f"Mean similarity across classes:               {cross:.4f}")

Training kernel, exact statevector simulation, shape [24, 24]
  min_eigenvalue           -7.28e-16
  max_asymmetry             4.44e-16
  max_diagonal_error        1.78e-15
  eigenvalue_check_passed  True (tolerance -1e-10)
  off_diagonal_mean        0.2213
  off_diagonal_std         0.3093
  off_diagonal_min         0.0000
  off_diagonal_max         1.0000
  identical_example_pairs  4
Mean similarity within a class (off-diagonal): 0.2453
Mean similarity across classes:               0.1993


In [12]:
i_idx, j_idx = np.nonzero(np.triu(np.isclose(K, 1.0, atol=1e-12), k=1))
pairs = pd.DataFrame(
    {
        "row_index a": train["row_index"].to_numpy()[i_idx],
        "row_index b": train["row_index"].to_numpy()[j_idx],
        "label a": y[i_idx],
        "label b": y[j_idx],
        "identical features": [
            np.array_equal(train_matrix[i], train_matrix[j])
            for i, j in zip(i_idx, j_idx, strict=True)
        ],
        "K": K[i_idx, j_idx],
    }
)
print("Training pairs with K = 1:")
pairs

Training pairs with K = 1:


,row_index a,row_index b,label a,label b,identical features,K
0,23411,119739,0,0,True,1.0
1,45541,124671,1,1,True,1.0
2,45541,214754,1,1,True,1.0
3,124671,214754,1,1,True,1.0


In [13]:
ordered = np.argsort(y, kind="stable")
names = np.array(["L", "P"])[y[ordered]]
print("Exact training kernel grouped by class (L = legitimate, P = phishing), 2 decimals:")
print(pd.DataFrame(K[np.ix_(ordered, ordered)], index=names, columns=names).round(2).to_string())

Exact training kernel grouped by class (L = legitimate, P = phishing), 2 decimals:
      L     L     L     L     L     L     L     L     L     L     L     L     P     P     P     P     P     P     P     P     P     P     P     P
L  1.00  0.98  0.63  0.99  0.00  0.84  0.24  0.01  0.84  0.00  0.43  0.97  0.79  0.01  0.00  0.09  0.34  0.61  0.00  0.03  0.03  0.09  0.71  0.09
L  0.98  1.00  0.49  0.93  0.00  0.93  0.29  0.00  0.93  0.00  0.41  0.91  0.65  0.01  0.00  0.03  0.31  0.72  0.00  0.02  0.03  0.03  0.57  0.03
L  0.63  0.49  1.00  0.74  0.00  0.24  0.00  0.42  0.24  0.00  0.29  0.77  0.97  0.06  0.00  0.64  0.25  0.10  0.00  0.04  0.01  0.64  0.99  0.64
L  0.99  0.93  0.74  1.00  0.00  0.75  0.19  0.04  0.75  0.00  0.43  1.00  0.87  0.02  0.00  0.15  0.35  0.51  0.00  0.03  0.02  0.15  0.81  0.15
L  0.00  0.00  0.00  0.00  1.00  0.00  0.01  0.00  0.00  0.78  0.03  0.00  0.00  0.02  0.00  0.00  0.00  0.00  0.58  0.01  0.23  0.00  0.00  0.00
L  0.84  0.93  0.24  0.75  0.00  1.00  0.

### Replication check: training kernel against the protocol

In [14]:
def documented(pattern):
    m = re.search(pattern, PROTOCOL)
    assert m, f"pattern not found in docs/benchmark-protocol.md: {pattern}"
    return m


# Validity is checked against tolerances; the recorded -7.3e-16 is floating-point noise and may differ by platform.
assert diag["shape"] == [24, 24]
assert diag["max_asymmetry"] <= 1e-12 and diag["max_diagonal_error"] <= 1e-12
assert diag["eigenvalue_check_passed"] and diag["min_eigenvalue"] >= -qkernel.EIGENVALUE_TOLERANCE
recorded_eig = float(documented(r"Minimum eigenvalue \| (-?[0-9.]+e-?\d+)")[1])
assert abs(diag["min_eigenvalue"]) < 1e-10 and abs(recorded_eig) < 1e-10
print(
    f"OK: symmetric, unit diagonal, min eigenvalue {diag['min_eigenvalue']:.1e} >= -1e-10 (recorded {recorded_eig:.1e}; both rounding noise around 0)."
)

m = documented(r"Off-diagonal mean and standard deviation \| ([0-9.]+) and ([0-9.]+)")
assert round(diag["off_diagonal_mean"], 3) == float(m[1]) and round(
    diag["off_diagonal_std"], 3
) == float(m[2])
m = documented(r"Off-diagonal range \| ([0-9.]+) to ([0-9.]+)")
assert round(diag["off_diagonal_min"], 3) == float(m[1]) and round(
    diag["off_diagonal_max"], 3
) == float(m[2])
print(
    f"OK: off-diagonal mean {diag['off_diagonal_mean']:.3f}, std {diag['off_diagonal_std']:.3f}, range {diag['off_diagonal_min']:.3f} to {diag['off_diagonal_max']:.3f} match the protocol."
)

n_pairs = int(
    documented(r"come from (\d+) pairs of training examples with identical feature vectors")[1]
)
assert diag["identical_example_pairs"] == n_pairs == len(pairs)
assert pairs["identical features"].all() and (pairs["label a"] == pairs["label b"]).all()
print(
    f"OK: {n_pairs} identical-example pairs, all with identical features and matching labels, as recorded."
)

m = documented(r"mean similarity is ([0-9.]+) within a class and ([0-9.]+) across classes")
assert round(within, 3) == float(m[1]) and round(cross, 3) == float(m[2])
print(
    f"OK: within-class {within:.3f} vs cross-class {cross:.3f} mean similarity match the protocol ({m[1]} vs {m[2]})."
)

saved_kernel = Path("artifacts/benchmark/training-kernel.npy")
if saved_kernel.exists():
    assert np.abs(np.load(saved_kernel) - K).max() <= 1e-9
    print(f"OK: kernel equals {saved_kernel.as_posix()} from the training run within 1e-9.")
else:
    print(
        f"SKIPPED: {saved_kernel.as_posix()} not present yet (created by scripts/train_benchmark.py)."
    )

OK: symmetric, unit diagonal, min eigenvalue -7.3e-16 >= -1e-10 (recorded -7.3e-16; both rounding noise around 0).
OK: off-diagonal mean 0.221, std 0.309, range 0.000 to 1.000 match the protocol.
OK: 4 identical-example pairs, all with identical features and matching labels, as recorded.
OK: within-class 0.245 vs cross-class 0.199 mean similarity match the protocol (0.245 vs 0.199).
SKIPPED: artifacts/benchmark/training-kernel.npy not present yet (created by scripts/train_benchmark.py).


## 5. From counts to a kernel value (local illustration)

On hardware, `K(x, y)` is estimated as the fraction of shots that return `0000` (`kernel_from_counts`). Here a **local** Qiskit `StatevectorSampler` runs one pair circuit at increasing shot counts. It is exact simulation plus multinomial shot sampling with a fixed seed, and it contacts no IBM service. First, the pair circuit's exact all-zero probability must equal the overlap formula. Then the shot estimate should approach the exact value, with binomial error `sqrt(K (1 - K) / shots)`; 1,024 shots is the protocol's per-pair setting. Near `K = 0.5`, one binomial standard deviation at 1,024 shots is about 1.6 points, so a single estimate a few points off (2 to 3 sigma) is within the expected spread. The seed is fixed and was not chosen to make the table look better; the cell prints how far the 1,024-shot estimate lands in this run. This illustrates the estimator only, not hardware noise.

In [15]:
# Pick the off-diagonal training pair whose exact K is closest to 0.5, where shot variance is largest.
upper = np.triu_indices(len(K), k=1)
best = int(np.argmin(np.abs(K[upper] - 0.5)))
a, b = int(upper[0][best]), int(upper[1][best])
exact = float(K[a, b])
circuit = qkernel.pair_circuit(train_angles[a], train_angles[b])

p_zero = float(
    Statevector(circuit.remove_final_measurements(inplace=False))
    .probabilities_dict()
    .get("0000", 0.0)
)
assert abs(p_zero - exact) <= 1e-9
print(
    f"Pair: training positions {a} and {b}. Exact K = {exact:.6f}; P(0000) of the pair circuit = {p_zero:.6f}"
)
print("OK: the compute-uncompute probability equals the statevector overlap (exact simulation).")

sampler = StatevectorSampler(seed=2026)
rows = []
for shots in (256, 1024, 16384, 262144):
    counts = sampler.run([circuit], shots=shots).result()[0].data.meas.get_counts()
    estimate = qkernel.kernel_from_counts(counts)
    sigma = math.sqrt(exact * (1 - exact) / shots)
    rows.append(
        {
            "shots": shots,
            "estimate": estimate,
            "abs error": abs(estimate - exact),
            "binomial sigma": sigma,
            "error / sigma": abs(estimate - exact) / sigma,
        }
    )
convergence = pd.DataFrame(rows)
assert (convergence["error / sigma"] < 5).all()
assert convergence["abs error"].iloc[-1] < 0.01
print(
    "OK: shot estimates stay within 5 sigma and approach the exact value (local sampler, not hardware)."
)
at_protocol = convergence.set_index("shots").loc[1024]
print(
    f"In this run the 1,024-shot estimate is off by {100 * at_protocol['abs error']:.1f} points "
    f"({at_protocol['error / sigma']:.1f} sigma), within the expected binomial spread."
)
convergence.round(5)

Pair: training positions 3 and 17. Exact K = 0.508497; P(0000) of the pair circuit = 0.508497
OK: the compute-uncompute probability equals the statevector overlap (exact simulation).


OK: shot estimates stay within 5 sigma and approach the exact value (local sampler, not hardware).
In this run the 1,024-shot estimate is off by 4.0 points (2.5 sigma), within the expected binomial spread.


,shots,estimate,abs error,binomial sigma,error / sigma
0,256,0.50781,0.00068,0.03125,0.02192
1,1024,0.46875,0.03975,0.01562,2.54421
2,16384,0.50873,0.00023,0.00391,0.05903
3,262144,0.51057,0.00207,0.00098,2.12309


## Interpretation

- **Scaling is reproducible.** The documented script regenerates `configs/feature-scaler.json` from the 24 training rows alone. Four test values fall outside the training range and are clipped: two hosts with more labels than any training host, one entropy value, and one digit ratio, all in phishing test rows. That is expected with only 24 training rows, and the counts were recorded before any model was evaluated.
- **The features are coarse.** `host_label_count` is binary in training, and the digit ratio and hyphen count are zero for most hosts. As a result, 4 training pairs have identical feature vectors and a kernel value of exactly 1.
- **The exact training kernel is valid.** It is symmetric, has a unit diagonal, and is positive semidefinite up to floating-point noise, so no correction is applied before fitting the SVM.
- **It is not concentrated, but it separates the classes only weakly.** The off-diagonal values have mean 0.221 and standard deviation 0.309 over the full 0 to 1 range, so the examples are distinguishable. Mean similarity is 0.245 within a class against 0.199 across classes. This is consistent with the quantum-kernel SVM's chance-level test accuracy recorded in the protocol. It reflects these features and this encoding on this split, not hardware noise, and it is not evidence about quantum kernels in general.
- **Limitations.** One split, one seed, 24 training and 12 test examples; each test example is worth 8.3 points of accuracy. Everything here is classical preprocessing or exact simulation, and the shot-sampling cell only illustrates the estimator. No quantum advantage is claimed, and nothing was tuned.

Next: training the four models and evaluating them, first on the exact simulated test kernel.